### Preprocessing step 2

In this step we aggregate the wind turbine data over all time steps per year.

After execution of this notebook, the origin folders of SCADA data are obsolete and will be deleted


In [1]:
import pandas as pd
import os
from glob import glob
from pathlib import Path
import shutil
from tqdm import tqdm
from helperfunctions.intern_constants import WT_ID
from helperfunctions import intern_constants as ic

#### The execution of the following code might take ~24 min
DO NOT CANCEL THIS ONCE IT IS RUNNING!

In [3]:
path = Path(ic.PATH_PROJECT_ROOT / "src" / "Penmanshiel")

all_wt_data = []
all_status_logs = []

for year_folder in tqdm(os.listdir(path)):
    folder_path = os.path.join(path, year_folder)
    if not os.path.isdir(folder_path):
        continue
    
    csv_files = glob(os.path.join(folder_path, "*.csv"))
    
    for file in tqdm(csv_files):
        filename = os.path.basename(file)
        try: 
            if filename.startswith("Turbine_Data"):
                turbine_id = filename.split("_")[3]
            elif filename.startswith("Status"): 
                turbine_id = filename.split("_")[2]
            else:
                print(f"unknown file: {filename}")
                continue
            turbine_id = int(turbine_id)
        except (IndexError, ValueError):
            print(f"Parsing error ID: {filename}")
            continue

        try:
            df = pd.read_csv(file, skiprows=9)
            df.columns = [col.lstrip('#').strip() for col in df.columns]
            df[WT_ID] = turbine_id
            
            if filename.startswith("Turbine_Data"):
                all_wt_data.append(df)
            elif filename.startswith("Status"):
                all_status_logs.append(df)
        except Exception as e:
            print(f"Error in reading file= {file}: {e}")

df_turbines = pd.concat(all_wt_data, ignore_index=True)
df_status = pd.concat(all_status_logs, ignore_index=True)


for turbine_id in tqdm(df_turbines[WT_ID].unique()):
    df_wt = df_turbines[df_turbines[WT_ID]==turbine_id]
    df_wt.to_csv(os.path.join(path, f"WT_Data_Complete_ID_{turbine_id}.csv"), index=False)

for turbine_id in tqdm(df_status[WT_ID].unique()):
    df_sl = df_status[df_status[WT_ID]==turbine_id]
    df_sl.to_csv(os.path.join(path, f"Status_Logs_Complete_ID_{turbine_id}.csv"), index=False)


100%|██████████| 14/14 [00:03<00:00,  4.59it/s]


In [4]:
path = Path(ic.PATH_PROJECT_ROOT / "src" / "Penmanshiel")

all_wt_data = []
all_status_logs = []

for year_folder in tqdm(os.listdir(path)):
    path_to_delete = os.path.join(path,year_folder)
    if Path(path_to_delete).exists() and Path(path_to_delete).is_dir():
            shutil.rmtree(path_to_delete)
   
   

100%|██████████| 35/35 [00:00<00:00, 528.57it/s]
